<a href="https://colab.research.google.com/github/shivansh2310/Quantitative-Momentum/blob/main/Seasonality_%26_Window_Dressing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A. Window Dressing (The Institutional Mark-Up)

Active mutual fund managers have to report their holdings to investors at the end of every quarter (March, June, September, December). If a manager has been losing money, they don't want to show their clients a portfolio full of losers.

* The Dumb Money Move: In the last week of the quarter, they aggressively sell their losers and buy the quarter's biggest winners, so their 13F filings look "smart."

* The Quant Move: This creates artificial, predictable buying pressure on momentum stocks at the end of the quarter. If you rebalance your portfolio on the last day of the quarter, you are buying after the price has been artificially inflated. You want to front-run them.

### B. The Rebalancing Premium (The Offset)

Research shows that by simply shifting your rebalance date from the last trading day of the month to roughly the 25th day of the month (or 5 trading days prior to month-end), you capture the "Rebalancing Premium." You buy the momentum names just before the mutual funds start their window dressing, and you let their buying pressure push your portfolio up.

### C. The January Effect (Momentum's Kryptonite)

In December, retail and institutional investors dump losing stocks to harvest tax write-offs. In January, that selling pressure vanishes, and the beaten-down losers violently bounce back. Momentum strategies historically perform terribly in January.

## Seasonal Rebalance Engine

In [14]:
import pandas as pd
import numpy as np
import datetime
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.tseries.offsets import BMonthEnd, BDay

In [15]:
universe = [
    'AAPL', 'MSFT', 'AMZN', 'NVDA', 'META', 'TSLA', 'GOOGL',
    'JPM', 'BAC', 'GS', 'MS', 'WFC', 'C',
    'XOM', 'CVX', 'COP', 'EOG', 'SLB',
    'JNJ', 'UNH', 'LLY', 'PFE', 'ABBV',
    'PG', 'KO', 'PEP', 'WMT', 'TGT',
    'CAT', 'DE', 'GE', 'HON', 'MMM'
]
market_proxy = 'SPY'

# Combine for downloading
download_list = universe + [market_proxy]

In [16]:
data = yf.download(download_list, period="2y")['Close']
data = data.dropna(axis=1)

/tmp/ipykernel_7805/60239374.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(download_list, period="2y")['Close']
[*********************100%***********************]  34 of 34 completed


In [17]:
spy_close = data[market_proxy]
latest_date = spy_close.index[-1]

In [18]:
print("Initializing the Seasonal Rebalance Engine...")

# In a production environment, 'today' is pd.Timestamp.today()
current_date = pd.to_datetime(latest_date)

Initializing the Seasonal Rebalance Engine...


In [19]:
current_date

Timestamp('2026-06-23 00:00:00')

In [20]:
def calculate_optimal_rebalance_date(eval_date, offset_days=5):
    """
    Calculates the optimal rebalancing date to front-run month-end window dressing.
    offset_days: Number of business days BEFORE the end of the month to execute.
    """
    # Find the last business day of the CURRENT month
    month_end = eval_date + BMonthEnd(0)

    # If we are already past the month-end for some reason, look to the next month
    if eval_date > month_end:
        month_end = eval_date + BMonthEnd(1)

    # Subtract the optimal offset (e.g., 5 business days)
    optimal_execution_date = month_end - BDay(offset_days)

    return optimal_execution_date.date(), month_end.date()

In [21]:
optimal_date, month_end_date = calculate_optimal_rebalance_date(current_date, offset_days=5)

print("\n" + "="*65)
print(" SEASONAL EXECUTION CALENDAR")
print("="*65)
print(f"Current System Date:             {current_date.date()}")
print(f"Standard Month-End Date:         {month_end_date}")
print("-" * 65)
print(f"OPTIMAL TARGET EXECUTION DATE:   {optimal_date}")
print("="*65)

# Actionable PM Output
days_to_execution = np.busday_count(current_date.date(), optimal_date)

print("\nPORTFOLIO MANAGER DIRECTIVE:")
if days_to_execution > 0:
    print(f"HOLD. We are {days_to_execution} business days away from the optimal rebalance window.")
    print("Executing today sacrifices the late-month liquidity premium.")
elif days_to_execution == 0:
    print("EXECUTE TODAY. 🟢")
    print("We are exactly 5 business days from month-end. Front-running institutional")
    print("window-dressing flows. Liquidate losers, allocate to High-Quality Momentum.")
else:
    print(f"MISSED WINDOW ({-days_to_execution} days past optimal).")
    print("We are inside the institutional execution window. You are paying the markup.")
    print("Hold current portfolio until the next month's offset window.")

# The January Warning
if current_date.month == 12 or current_date.month == 1:
    print("\n⚠️ SEASONALITY ALERT: THE JANUARY EFFECT")
    print("Momentum strategies historically face severe headwinds in January due to")
    print("tax-loss harvesting reversals. Consider reducing gross exposure or")
    print("tightening the Trend Filter (Day 2) during this period.")


 SEASONAL EXECUTION CALENDAR
Current System Date:             2026-06-23
Standard Month-End Date:         2026-06-30
-----------------------------------------------------------------
OPTIMAL TARGET EXECUTION DATE:   2026-06-23

PORTFOLIO MANAGER DIRECTIVE:
EXECUTE TODAY. 🟢
We are exactly 5 business days from month-end. Front-running institutional
window-dressing flows. Liquidate losers, allocate to High-Quality Momentum.
